# Bitonic Sort on PYNQ-Z2

Benchmarks the `sort_top` HLS IP against NumPy for **N = 64** 32-bit integers.

**Data path:** PS → DMA MM2S → AXI-Stream → `sort_top` → AXI-Stream → DMA S2MM → PS

| Register (s_axi_control) | Offset |
|--------------------------|--------|
| `ap_ctrl` (bit0=start, bit1=done, bit2=idle) | `0x00` |

## 1. Load Overlay

In [ ]:
import time
import numpy as np
from pynq import Overlay, allocate

N = 64

ol = Overlay('bitonic_design.bit')
print('Overlay loaded')
print('IP blocks:', list(ol.ip_dict.keys()))

dma      = ol.axi_dma_0
sort_ip  = ol.sort_top_0
hw_timer = ol.axi_timer_0

## 2. Helper Functions

In [ ]:
AP_CTRL = 0x00
TCSR0, TLR0, TCR0 = 0x00, 0x04, 0x08
FCLK_MHZ = 100.0


def timer_start(tmr):
    tmr.write(TLR0, 0)
    tmr.write(TCSR0, 0x020)  # load
    tmr.write(TCSR0, 0x080)  # enable, count up


def timer_stop(tmr):
    cycles = tmr.read(TCR0)
    tmr.write(TCSR0, 0x000)
    return cycles


def bitonic_sort_hw(data, timed=False):
    """Sort data using the FPGA bitonic sort IP via AXI DMA."""
    assert len(data) == N, f'IP is hardcoded for N={N}'

    in_buf  = allocate(shape=(N,), dtype=np.int32)
    out_buf = allocate(shape=(N,), dtype=np.int32)
    np.copyto(in_buf, data.astype(np.int32))
    out_buf[:] = 0

    if timed:
        timer_start(hw_timer)

    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    sort_ip.write(AP_CTRL, 0x01)  # ap_start

    dma.sendchannel.wait()
    dma.recvchannel.wait()

    cycles = timer_stop(hw_timer) if timed else None

    result = np.array(out_buf, dtype=np.int32)
    in_buf.freebuffer()
    out_buf.freebuffer()

    return (result, cycles) if timed else result

## 3. Correctness Check

In [ ]:
rng = np.random.default_rng(42)
all_pass = True

# Random trials
print('Random trials:')
for trial in range(5):
    data   = rng.integers(-2**30, 2**30, size=N, dtype=np.int32)
    hw_out = bitonic_sort_hw(data)
    ok     = np.array_equal(hw_out, np.sort(data))
    all_pass &= ok
    print(f'  trial {trial+1}: {"PASS" if ok else "FAIL"}')

# Edge cases
print('Edge cases:')
edge_cases = [
    ('ascending',  np.arange(N,        dtype=np.int32)),
    ('descending', np.arange(N-1, -1, -1, dtype=np.int32)),
    ('all-same',   np.full(N, 7,        dtype=np.int32)),
    ('all-neg',    rng.integers(-2**31+1, 0, size=N, dtype=np.int32)),
    ('mixed-sign', rng.integers(-500, 500, size=N, dtype=np.int32)),
]
for label, data in edge_cases:
    ok = np.array_equal(bitonic_sort_hw(data), np.sort(data))
    all_pass &= ok
    print(f'  {label:<12}: {"PASS" if ok else "FAIL"}')

print()
print('Overall:', 'PASS ✓' if all_pass else 'FAIL ✗')

## 4. Timing Benchmark

Two measurements per HW run:
- **Wall time** – `time.perf_counter()` around the full DMA + sort call (includes Python overhead)
- **HW cycles** – AXI timer counts PL clock cycles from DMA start to DMA completion

> Note: the AXI timer starts before the DMA transfer, so HW cycles includes DMA latency, not just sort latency.

In [ ]:
RUNS = 20
hw_cycles_list = []
wall_us_list   = []
sw_us_list     = []

for _ in range(RUNS):
    data = rng.integers(-2**30, 2**30, size=N, dtype=np.int32)

    # HW run
    t0 = time.perf_counter()
    _, cycles = bitonic_sort_hw(data, timed=True)
    wall_us_list.append((time.perf_counter() - t0) * 1e6)
    hw_cycles_list.append(cycles)

    # SW baseline (numpy)
    t0 = time.perf_counter()
    np.sort(data)
    sw_us_list.append((time.perf_counter() - t0) * 1e6)

hw_cycles = np.array(hw_cycles_list)
hw_us     = hw_cycles / FCLK_MHZ          # cycles → µs
wall_us   = np.array(wall_us_list)
sw_us     = np.array(sw_us_list)

print(f'N = {N}, runs = {RUNS}')
print()
print(f'{"Metric":<28} {"Mean":>10} {"Min":>10} {"Max":>10}')
print('-' * 62)
print(f'{"HW cycles (AXI timer)":<28} {hw_cycles.mean():>10.0f} {hw_cycles.min():>10.0f} {hw_cycles.max():>10.0f}')
print(f'{"HW time µs (AXI timer)":<28} {hw_us.mean():>10.2f} {hw_us.min():>10.2f} {hw_us.max():>10.2f}')
print(f'{"Wall time µs (w/ DMA+Python)":<28} {wall_us.mean():>10.2f} {wall_us.min():>10.2f} {wall_us.max():>10.2f}')
print(f'{"np.sort µs":<28} {sw_us.mean():>10.2f} {sw_us.min():>10.2f} {sw_us.max():>10.2f}')
print()
print(f'HW speedup vs np.sort (AXI timer): {sw_us.mean()/hw_us.mean():.2f}x')
print(f'HW speedup vs np.sort (wall time):  {sw_us.mean()/wall_us.mean():.2f}x')

## 5. Latency Breakdown

Theoretical minimum latency breakdown at 100 MHz:

| Stage | Cycles |
|-------|--------|
| DMA in (N=64 words) | 64 |
| Bitonic sort (21 sub-stages, II=1) | 21 |
| DMA out (N=64 words) | 64 |
| **Total** | **149** |

Expected: **1.49 µs** of pure HW time. Anything above that is DMA setup and bus overhead.

In [ ]:
theoretical_cycles = N + 21 + N   # in + sort + out
theoretical_us     = theoretical_cycles / FCLK_MHZ
overhead_us        = hw_us.mean() - theoretical_us

print(f'Theoretical minimum : {theoretical_cycles} cycles = {theoretical_us:.2f} µs')
print(f'Measured (AXI timer): {hw_cycles.mean():.0f} cycles = {hw_us.mean():.2f} µs')
print(f'DMA + bus overhead  : {overhead_us:.2f} µs  ({overhead_us/hw_us.mean()*100:.1f}% of total)')

## 6. Distribution Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Timing comparison bar chart
labels  = ['np.sort', 'HW (AXI timer)', 'HW (wall time)']
means   = [sw_us.mean(), hw_us.mean(), wall_us.mean()]
stds    = [sw_us.std(),  hw_us.std(),  wall_us.std()]
colors  = ['steelblue', 'seagreen', 'coral']

bars = axes[0].bar(labels, means, yerr=stds, capsize=5,
                   color=colors, alpha=0.8, edgecolor='black')
axes[0].set_ylabel('Time (µs)')
axes[0].set_title(f'Sort latency — N={N}')
axes[0].grid(axis='y', alpha=0.3)
for bar, mean in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{mean:.1f}', ha='center', va='bottom', fontsize=9)

# HW cycle distribution histogram
axes[1].hist(hw_cycles, bins=15, color='seagreen', alpha=0.8, edgecolor='black')
axes[1].axvline(theoretical_cycles, color='red', linestyle='--',
                label=f'Theoretical min ({theoretical_cycles})')
axes[1].set_xlabel('PL clock cycles')
axes[1].set_ylabel('Count')
axes[1].set_title('HW cycle count distribution')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()